# Exercise 2: Your First LangChain Chain

**Level:** Basic

In this exercise, you will build your first **LangChain chain** using the LangChain Expression Language (LCEL). You will learn how to connect prompts, models, and output parsers into a composable pipeline.

**What you will learn:**
- How to set up ChatGoogleGenerativeAI
- How to create prompt templates
- How to chain components with LCEL (`|` operator)
- How to parse structured output with Pydantic
## 1. Setup & Installation

In [ ]:
!pip install langchain langchain-google-genai pydantic -q
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

## 2. Your First LLM Call

Let's start with the simplest possible interaction — sending a message to an LLM and getting a response.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the model
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Send a simple message
response = model.invoke("What is the capital of France?")
print(response)
print(f"\nContent: {response.content}")
print(f"Type: {type(response)}")

Notice the response is an `AIMessage` object, not a plain string. LangChain wraps everything in message types for consistency.

## 3. Prompt Templates

Hardcoding prompts is fragile. **PromptTemplate** lets you create reusable prompts with variables.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt template with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant specializing in {region}."),
    ("human", "Suggest {count} must-visit places in {city}.")
])

# See what the prompt looks like with values filled in
formatted = prompt.invoke({
    "region": "Europe",
    "count": 3,
    "city": "Istanbul"
})
print("Formatted prompt:")
for msg in formatted.messages:
    print(f"  [{msg.type}]: {msg.content}")

## 4. Output Parsers

By default, the model returns an `AIMessage`. Parsers transform the output into the format you need.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser extracts just the text content
parser = StrOutputParser()

# Without parser
raw_response = model.invoke("Say hello in Turkish")
print(f"Without parser: {raw_response}")
print(f"Type: {type(raw_response)}")

# With parser
parsed_response = parser.invoke(raw_response)
print(f"\nWith parser: {parsed_response}")
print(f"Type: {type(parsed_response)}")

## 5. LCEL — Chaining with the Pipe Operator

**LangChain Expression Language (LCEL)** lets you compose components using the `|` (pipe) operator. This is the core pattern:

```python
chain = prompt | model | parser
```

Data flows left to right: prompt fills the template → model generates a response → parser extracts the text.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the components
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise travel guide. Answer in 2-3 sentences."),
    ("human", "What is {city} famous for?")
])

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()

# Build the chain with LCEL
chain = prompt | model | parser

# Run it!
result = chain.invoke({"city": "Istanbul"})
print(f"Istanbul: {result}")

print("---")

result = chain.invoke({"city": "Tokyo"})
print(f"Tokyo: {result}")

## 6. Streaming

LCEL chains automatically support streaming. Instead of waiting for the full response, you get tokens as they are generated.

In [ ]:
# Stream the response token by token
print("Streaming response for 'Paris':")
for chunk in chain.stream({"city": "Paris"}):
    print(chunk, end="", flush=True)
print()  # newline at the end

## 7. Batch Processing

LCEL also supports batch processing — send multiple inputs at once for parallel execution.

In [ ]:
# Process multiple cities at once
cities = [
    {"city": "Berlin"},
    {"city": "Barcelona"},
    {"city": "Amsterdam"}
]

results = chain.batch(cities)

for city_input, result in zip(cities, results):
    print(f"\n{city_input['city']}: {result}")

## 8. Multi-Step Chains

You can compose chains of chains. Let's build one that first summarizes a city and then translates the summary.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Step 1: Generate a city description
describe_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a travel writer. Write exactly one paragraph about the city."),
    ("human", "Describe {city} for a first-time visitor.")
])

# Step 2: Translate the description
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following text to {language}."),
    ("human", "{text}")
])

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()

# Chain step 1
describe_chain = describe_prompt | model | parser

# Full pipeline: describe → translate
full_chain = (
    {"text": describe_chain, "language": lambda x: x["language"]}
    | translate_prompt
    | model
    | parser
)

# Run the full pipeline
result = full_chain.invoke({"city": "Istanbul", "language": "Turkish"})
print(result)

## 9. Structured Output with Pydantic

Getting free-text back from an LLM is fine for chat, but for **agent development** we need structured, predictable output. LangChain integrates with Pydantic to enforce output schemas.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Define the output schema
class CityInfo(BaseModel):
    """Information about a city."""
    name: str = Field(description="City name")
    country: str = Field(description="Country the city is in")
    population_millions: float = Field(description="Approximate population in millions")
    top_attractions: list[str] = Field(description="Top 3 tourist attractions")
    best_season: str = Field(description="Best season to visit")

# Create the parser
pydantic_parser = PydanticOutputParser(pydantic_object=CityInfo)

# Create a prompt that includes the format instructions
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledgeable travel assistant. Always respond in the requested format."),
    ("human", "Give me information about {city}.\n\n{format_instructions}")
])

# Build the chain
structured_chain = (
    structured_prompt.partial(format_instructions=pydantic_parser.get_format_instructions())
    | model
    | pydantic_parser
)

# Run it
city_info = structured_chain.invoke({"city": "Istanbul"})

print(f"City: {city_info.name}")
print(f"Country: {city_info.country}")
print(f"Population: {city_info.population_millions}M")
print(f"Top attractions: {', '.join(city_info.top_attractions)}")
print(f"Best season: {city_info.best_season}")
print(f"\nType: {type(city_info)}")

## 10. with_structured_output (Preferred Approach)

LangChain also provides `with_structured_output()` which uses function calling under the hood — more reliable than parsing.

In [ ]:
from pydantic import BaseModel, Field

class FlightRecommendation(BaseModel):
    """A flight recommendation."""
    origin: str = Field(description="Departure city")
    destination: str = Field(description="Arrival city")
    reason: str = Field(description="Why this route is recommended")
    estimated_price_usd: int = Field(description="Rough price estimate in USD")
    best_month: str = Field(description="Best month to fly this route")

# Use with_structured_output for reliable structured responses
structured_model = model.with_structured_output(FlightRecommendation)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a flight booking assistant."),
    ("human", "Recommend a flight from {origin} for a {trip_type} trip.")
])

chain = prompt | structured_model

result = chain.invoke({"origin": "Istanbul", "trip_type": "beach vacation"})

print(f"Route: {result.origin} → {result.destination}")
print(f"Reason: {result.reason}")
print(f"Price: ~${result.estimated_price_usd}")
print(f"Best month: {result.best_month}")
print(f"\nType: {type(result)}")

---
## YOUR TURN: Exercise A

Build a chain that takes a **product name** and **target audience** as inputs and generates:
1. A marketing tagline (short, catchy)
2. A product description (2-3 sentences)

Use `with_structured_output` to return a Pydantic model with `tagline` and `description` fields.

In [ ]:
# YOUR TURN: Build the marketing chain

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# TODO: Define a Pydantic model for the output

# TODO: Create a prompt template

# TODO: Build the chain with with_structured_output

# TODO: Test with at least 2 different inputs

# Example test:
# result = chain.invoke({"product": "Smart Water Bottle", "audience": "fitness enthusiasts"})
# print(f"Tagline: {result.tagline}")
# print(f"Description: {result.description}")

---
## YOUR TURN: Exercise B

Build a **multi-step chain** that:
1. Takes a topic and generates 3 quiz questions about it (structured output: list of questions)
2. Takes those questions and generates answers for each one
3. Returns a final structured object with `topic`, `questions`, and `answers`

This exercises chaining multiple LLM calls together.

In [ ]:
# YOUR TURN: Build the quiz generator chain

# TODO: Define Pydantic models for quiz questions and answers

# TODO: Build step 1 — generate questions

# TODO: Build step 2 — generate answers

# TODO: Chain them together

# TODO: Test with a topic like "Python programming" or "Machine Learning"

## Key Takeaways

- **ChatGoogleGenerativeAI** is LangChain's wrapper around Google Gemini models
- **ChatPromptTemplate** creates reusable prompt templates with variables
- **LCEL** (`|` operator) chains components: `prompt | model | parser`
- **StrOutputParser** extracts text; **PydanticOutputParser** extracts structured data
- **`with_structured_output()`** is the most reliable way to get structured responses
- Chains automatically support `.stream()` and `.batch()` for free

**Next:** In Exercise 3, we will add memory so our chains can remember previous conversations.